# LookAhead-GateSkip: Colab runner

One notebook for every GPU session (docs/GPU_RUNBOOK.md, section "Google Colab"). Run the **Setup** cells top to bottom at the start of each session, then only the job cell(s) for that session.

| Session | GPU | Jobs |
|---|---|---|
| S1 | T4 (free) | setup, `pytest`, T-EQ, 50-problem smoke test, CUDA speed check |
| S2 | L4 | Dense-FT training (P3-07), then the P3-08 format check on the calibration split |
| S3 | L4 | GateSkip port training + calibration, pilot E1–E3 |
| S4 | L4 | Our method training + calibration, E4 |

**Rules**
- **T4 = fp16**, used only for tests, smoke runs and speed checks. **Training waits for an L4 or A100** (bf16); the training cells refuse to run on a T4.
- Tokens come from **Colab Secrets** (key icon in the left bar): `HF_TOKEN` and `GITHUB_TOKEN` (fine-grained, *Contents: read and write* on this repo only). They are never printed or written to a file.
- Persistent data lives in Google Drive under `MyDrive/lagate/`: the HF cache, checkpoints, raw generations and a mirror of `results/`. The repo and venv are rebuilt on the fast local disk each session (≈ 5 min).
- Only small result files (`config.yaml`, `meta.json`, `metrics.json`, thresholds) are committed and pushed. Raw outputs and checkpoints stay on Drive.
- Training checkpoints every 30 min and **resumes automatically**: if the session dies, run Setup again, then the same training cell.

## Setup (every session)

In [ ]:
# Session settings
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/lagate"      # persistent: HF cache, checkpoints, raw outputs, results mirror
REPO_DIR = "/content/lookahead-gateskip"          # ephemeral, fast local disk
CUDA_TAG = "cu128"                                # torch wheel index; lower it (e.g. cu126) if `nvidia-smi` shows an older CUDA
GIT_NAME = "Rishabh Nagar (Colab)"
GIT_EMAIL = "rishabh-nagar-9100@users.noreply.github.com"

In [ ]:
# Mount Drive, read secrets into this kernel's environment (never printed), set persistent paths
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
for sub in ("hf_cache", "checkpoints", "raw", "results_mirror"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

for name in ("HF_TOKEN", "GITHUB_TOKEN"):
    try:
        val = userdata.get(name)
    except Exception:
        val = None
    if val:
        os.environ[name] = val
    print(f"{name}: {'set' if val else 'MISSING (add it in Colab Secrets and grant this notebook access)'}")
del val

os.environ["HF_HOME"] = f"{DRIVE_ROOT}/hf_cache"
os.environ["LAGATE_CHECKPOINT_ROOT"] = f"{DRIVE_ROOT}/checkpoints"
os.environ["CUDA_TAG"] = CUDA_TAG
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["HF_DATASETS_OFFLINE"] = "0"

In [ ]:
# Clone or update the repo (token via an inline credential helper, never stored), link Drive folders
import subprocess, sys

def _bootstrap_git(*args):
    helper = '!f() { echo username=x-access-token; echo "password=${GITHUB_TOKEN}"; }; f'
    r = subprocess.run(["git", "-c", "credential.helper=", "-c", f"credential.helper={helper}", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git failed: " + r.stderr.replace(os.environ.get("GITHUB_TOKEN", "\0"), "<GITHUB_TOKEN>"))

if not os.path.exists(f"{REPO_DIR}/.git"):
    _bootstrap_git("clone", "--branch", BRANCH, "https://github.com/rishabh-nagar-9100/lookahead-gateskip.git", REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/scripts")
import importlib, colab_helpers as ch
importlib.reload(ch)

ch.clone_or_pull(REPO_DIR, BRANCH)
print(ch.link_to_drive(REPO_DIR, DRIVE_ROOT))
ch.sh(["git", "config", "user.name", GIT_NAME], cwd=REPO_DIR)
ch.sh(["git", "config", "user.email", GIT_EMAIL], cwd=REPO_DIR)
ch.sh("git log -1 --oneline", cwd=REPO_DIR)

In [ ]:
# GPU: bf16 on L4/A100, fp16 on T4. Every run's meta.json records the GPU (lagate.utils.accelerator_info).
GPU = ch.gpu_info()
print(GPU)
TRAINING_ALLOWED = GPU.get("training_allowed") == "True"
if not TRAINING_ALLOWED:
    print("This GPU has no native bf16 (e.g. T4): tests, smoke runs and speed checks only. Training cells will refuse.")

In [ ]:
# Environment: .venv with Python 3.11, CUDA torch (CUDA_TAG) + requirements.lock.txt (setup_gpu.sh env, ≈ 3-5 min)
if not os.path.exists(f"{REPO_DIR}/.venv/bin/python"):
    ch.sh("bash scripts/setup_gpu.sh env", cwd=REPO_DIR)
else:
    ch.sh(".venv/bin/python -c 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))'", cwd=REPO_DIR)

def run(cmd):
    # Run a repo command inside the venv (tokens come from the environment; nothing is printed)
    return ch.sh(f"source .venv/bin/activate && {cmd}", cwd=REPO_DIR)

def finish(message):
    # Mirror results/ to Drive, then commit + push only the small result files
    ch.mirror_results(REPO_DIR, DRIVE_ROOT)
    files = ch.commit_and_push(REPO_DIR, message, BRANCH)
    print(f"pushed {len(files)} files")

## Setup stages (S1 on a T4; repeat `teq` on the first L4 session)
`scripts/setup_gpu.sh` stages, one cell each. `teq` runs `scripts/real_model_checks.py` with `configs/checks/qwen_cuda.yaml` (sizes, full-scale T-EQ, greedy match, bs=1 latency). `smoke` is the 50-problem GSM8K run with the D-08 config.

In [ ]:
run("bash scripts/setup_gpu.sh test")

In [ ]:
run("bash scripts/setup_gpu.sh teq")
finish("P0-13: Colab T-EQ + speed check (" + GPU["gpu"] + ")")

In [ ]:
run("bash scripts/setup_gpu.sh smoke")
finish("P0-13: Colab 50-problem smoke test (" + GPU["gpu"] + ")")

## Jobs (one cell per job, each driven by a config path)

### Smoke test (any GPU)

In [ ]:
SMOKE_CONFIG = "configs/eval/gsm8k_dense_qwen.yaml"   # D-08 reference: Qwen 62.0% on these 50 problems (MPS, run 20261106-182046)
run(f"python scripts/eval_dense.py --config {SMOKE_CONFIG} --seed 0 --limit 50")
finish(f"P0-13: smoke {SMOKE_CONFIG} on {GPU['gpu']}")

### CUDA speed check: HF generate vs our dense engine vs the routing path (bs=1)

In [ ]:
SPEED_CONFIG = "configs/checks/qwen_cuda.yaml"   # dtype auto: bf16 on L4/A100, fp16 on T4
run(f"python scripts/real_model_checks.py --config {SPEED_CONFIG} --seed 0")
finish(f"P5-01/P5-06: CUDA speed check on {GPU['gpu']}")

### Training (L4/A100 only): Dense-FT (S2), GateSkip port (S3), ours (S4)
Re-running the same cell after a disconnect resumes from `MyDrive/lagate/checkpoints/<name>_s<seed>/latest.pt`. Dense-FT also writes a merged HF model to `.../hf` for evaluation.

In [ ]:
TRAIN_CONFIG = "configs/train/denseft_qwen.yaml"   # S2: Dense-FT (P3-07). S3/S4: the GateSkip / ours configs
SEED = 1
assert TRAINING_ALLOWED, f"training needs an L4/A100 (bf16); this is {GPU['gpu']}"
run(f"python scripts/train.py --config {TRAIN_CONFIG} --seed {SEED}")
finish(f"P3-07: train {TRAIN_CONFIG} seed {SEED} on {GPU['gpu']}")

### Calibration (gated methods, after their training has finished)

In [ ]:
CALIB_CONFIG = "configs/calibrate/<method>_qwen.yaml"   # S3/S4: written with the gated training configs
SEED = 1
run(f"python scripts/calibrate.py --config {CALIB_CONFIG} --seed {SEED}")
finish(f"P3-04: calibrate {CALIB_CONFIG} seed {SEED}")

### Evaluation in shards
Each shard is its own run and fits in one session. Shards already finished with the same code are skipped, so this cell can be re-run in a later session until all are done; then run the merge cell. All shards of one evaluation must run on the same GPU type (the merge checks it).

In [ ]:
EVAL_CONFIG = "configs/eval/gsm8k_chat_denseft_qwen_calib.yaml"   # S2: P3-08 check on the calibration split; later the test-set configs
SEED = 0
NUM_SHARDS = 4
SHARDS_THIS_SESSION = [0, 1, 2, 3]

import yaml
name = yaml.safe_load(open(f"{REPO_DIR}/{EVAL_CONFIG}"))["name"]
for i in SHARDS_THIS_SESSION:
    if i in ch.completed_shards(REPO_DIR, name, NUM_SHARDS, SEED):
        print(f"shard {i}/{NUM_SHARDS}: already done, skipping")
        continue
    run(f"python scripts/eval_dense.py --config {EVAL_CONFIG} --seed {SEED} --shard {i} --num-shards {NUM_SHARDS}")
    finish(f"P4: eval {name} shard {i}/{NUM_SHARDS} on {GPU['gpu']}")
print("done:", sorted(ch.completed_shards(REPO_DIR, name, NUM_SHARDS, SEED)))

In [ ]:
# Merge the shards (refuses missing / overlapping shards, different code, GPU or dtype)
run(f"python scripts/merge_shards.py --config {EVAL_CONFIG} --seed {SEED} --glob 'results/*_{name}_shard*of{NUM_SHARDS}_s{SEED}'")
finish(f"P4: merge {name} ({NUM_SHARDS} shards)")

In [ ]:
# P3-08 verification checkpoint (D-15) on the merged calibration-split run: all three must pass
import glob, json
merged = sorted(glob.glob(f"{REPO_DIR}/results/*_{name}_merged{NUM_SHARDS}_s{SEED}/metrics.json"))[-1]
m = json.load(open(merged)); t = next(iter(m["results"]))
strict, flex = m["results"][t]["exact_match,strict-match"], m["results"][t]["exact_match,flexible-extract"]
checks = {
    ">= 90% '#### N'": m["format"][t]["frac_hash_answer"] >= 0.90,
    "<= 2% hit the 512-token cap": m["output_length"][t]["frac_hit_max_gen_toks"] <= 0.02,
    "|strict - flexible| <= 4 pts": abs(strict - flex) * 100 <= 4,
}
print(merged)
print(f"strict {strict:.3f}  flexible {flex:.3f}  robust {m['robust_extract'][t]['accuracy']:.3f}  "
      f"#### {m['format'][t]['frac_hash_answer']:.3f}  cap {m['output_length'][t]['frac_hit_max_gen_toks']:.3f}")
for k, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + k)
print("P3-08:", "PASSED" if all(checks.values()) else "FAILED -> revisit the protocol before any gated training (D-15)")

## Manual commit (if a job cell stopped before `finish`)

In [ ]:
finish("results from Colab session on " + GPU["gpu"])